# Fase 2: Preprocesamiento y Unificación de Datos

Siguiendo el flujo de trabajo de Machine Learning propuesto por Aurélien Géron, una vez finalizada la fase de Exploración (EDA), procedemos a la fase de **Preparación de Datos (Data Preparation)**.

El objetivo de este notebook es ejecutar sistemáticamente los pasos de **Data Cleaning** y **Feature Engineering** para unificar 8 archivos crudos y generar **DOS datasets procesados** y listos para entrenar algoritmos de clasificación.

## 1. Configuración de Entorno
Importamos las librerías esenciales y centralizamos las rutas de los archivos (incluyendo la data externa de ubigeos) desde nuestro módulo de configuración `config.py`.

In [ ]:
import pandas as pd
import numpy as np
import sys
from pathlib import Path

# Importar rutas desde el config maestro
sys.path.append('../src')
from config import (
    DIR_SIEN_REGIONAL, 
    FILE_ANEMIA_DA, 
    FILE_TAMIZAJE, 
    REGIONES_SELECCIONADAS,
    PROCESSED_DIR,
    FILE_UBIGEO
)

# Asegurarnos de que el directorio de salida existe
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Configuraciones visuales para el notebook
pd.set_option('display.max_columns', None)

## 2. Data Cleaning: Carga y Deduplicación
El primer paso de limpieza es la eliminación de redundancias. Cargamos los 8 archivos de salud crudos y ejecutamos `.drop_duplicates()` instantáneamente para evitar sobreajuste (*overfitting*) generado por registros clonados por error del sistema.

In [ ]:
print("Iniciando carga y limpieza de duplicados...")
dfs_sien = []

# 1. Cargar Regiones SIEN y eliminar duplicados inmediatamente
for region in REGIONES_SELECCIONADAS:
    df_temp = pd.read_csv(DIR_SIEN_REGIONAL / region, low_memory=False)
    duplicados_antes = len(df_temp)
    df_temp = df_temp.drop_duplicates()
    print(f"{region}: {duplicados_antes - len(df_temp)} duplicados eliminados.")
    dfs_sien.append(df_temp)

# 2. Cargar ANEMIA_DA y TAMIZAJE
df_da = pd.read_csv(FILE_ANEMIA_DA, sep=';', low_memory=False).drop_duplicates()
df_tamizaje = pd.read_csv(FILE_TAMIZAJE, low_memory=False).drop_duplicates()
print(f"ANEMIA_DA cargado. Shape: {df_da.shape}")
print(f"TAMIZAJE cargado. Shape: {df_tamizaje.shape}")

# 3. Cargar diccionario de Ubigeos para Feature Engineering posterior
df_ubigeo = pd.read_csv(FILE_UBIGEO, low_memory=False)

## 3. Data Formatting: Armonización de Atributos
Para poder concatenar las fuentes, los atributos deben compartir el mismo espacio de nombres. Mapeamos las cabeceras incompatibles de los datasets nacionales (`ANEMIA_DA` y `TAMIZAJE`) al estándar del MINSA-SIEN regional.

In [ ]:
# Renombramos las columnas de los archivos nacionales

map_da = {
    'EDAD_REGISTRO': 'EdadMeses',
    'GENERO': 'Sexo',
    'DIAGNOSTICO': 'Dx_anemia',
    'LATITUD': 'Latitud_Origen',
    'LONGITUD': 'Longitud_Origen'
}
df_da = df_da.rename(columns=map_da)

map_tamizaje = {
    'Edad': 'EdadMeses',
    'Diagnostico': 'Dx_anemia'
}
df_tamizaje = df_tamizaje.rename(columns=map_tamizaje)

print("Cabeceras de ANEMIA_DA y TAMIZAJE armonizadas al estandar SIEN.")

## 4. Data Cleaning: Filtros de Alcance y Manejo de Outliers
Restringimos el alcance del proyecto a niños **menores de 5 años** (<= 59.99 meses). Además, aplicamos limpieza sobre la variable predictora principal (Hemoglobina) descartando *Outliers* o valores biológicamente imposibles que puedan sesgar al modelo.

In [ ]:
# 1. Corrección especial para TAMIZAJE: Convertir Edad de años a meses si Tipo_edad es 'A'
if 'Tipo_edad' in df_tamizaje.columns:
    filtro_anos = df_tamizaje['Tipo_edad'] == 'A'
    df_tamizaje.loc[filtro_anos, 'EdadMeses'] = df_tamizaje.loc[filtro_anos, 'EdadMeses'] * 12

def aplicar_filtros(df, nombre):
    inicial = len(df)
    
    # Asegurar que EdadMeses sea numérico
    df['EdadMeses'] = pd.to_numeric(df['EdadMeses'], errors='coerce')
    
    # Filtro de alcance: Menores de 5 años (<= 59.99 meses)
    df = df[df['EdadMeses'] <= 59.99]
    
    # Tratamiento de Outliers en Hemoglobina
    if 'Hemoglobina' in df.columns:
        df['Hemoglobina'] = pd.to_numeric(df['Hemoglobina'], errors='coerce')
        # Retener solo valores biológicamente viables (ej. entre 4 y 20 g/dL)
        df = df[(df['Hemoglobina'] >= 4) & (df['Hemoglobina'] <= 20)]
        
    print(f"Filtros en {nombre}: Retenidos {len(df)} de {inicial} registros.")
    return df

for i, df_temp in enumerate(dfs_sien):
    dfs_sien[i] = aplicar_filtros(df_temp, f"SIEN Región {i+1}")

df_da = aplicar_filtros(df_da, "ANEMIA_DA")
df_tamizaje = aplicar_filtros(df_tamizaje, "TAMIZAJE")

## 5. Handling Categorical Attributes: Transformación del Target
Los algoritmos matemáticos requieren formatos estandarizados. Transformamos las etiquetas categóricas dispares (Textos descriptivos del SIEN y Códigos CIE-10 del Tamizaje) en una variable **Target Binaria** uniforme:
* `0`: Sano / Normal
* `1`: Riesgo de Anemia

In [ ]:
def estandarizar_target(val):
    val = str(val).strip().upper()
    if pd.isna(val) or val == 'NAN':
        return np.nan
    
    # Casos Normales / Sanos
    if 'NORMAL' in val:
        return 0
        
    # Casos de Anemia (Textos o Códigos como 85018.00)
    if 'ANEMIA' in val or '85018' in val:
        return 1
        
    return np.nan

# Aplicar a SIEN
for df_temp in dfs_sien:
    df_temp['Target'] = df_temp['Dx_anemia'].apply(estandarizar_target)

# Aplicar a Nacionales
df_da['Target'] = df_da['Dx_anemia'].apply(estandarizar_target)
df_tamizaje['Target'] = df_tamizaje['Dx_anemia'].apply(estandarizar_target)

print("Target Binario estandarizado en todos los datasets.")

## 6. Feature Engineering: Enriquecimiento Socioespacial
En lugar de depender exclusivamente de los datos clínicos, generamos nuevas características predictivas (*Feature Engineering*) inyectando índices socioeconómicos y altitud desde el padrón del INEI. Esto dota al modelo de un fuerte contexto geodemográfico.

In [ ]:
# Estandarizamos ubigeo base para el cruce
df_ubigeo['inei'] = pd.to_numeric(df_ubigeo['inei'], errors='coerce')
df_ubigeo['reniec'] = pd.to_numeric(df_ubigeo['reniec'], errors='coerce')
df_ubigeo['departamento'] = df_ubigeo['departamento'].str.upper().str.strip()
df_ubigeo['provincia'] = df_ubigeo['provincia'].str.upper().str.strip()
df_ubigeo['distrito'] = df_ubigeo['distrito'].str.upper().str.strip()

columnas_a_inyectar = ['altitude', 'latitude', 'longitude', 'idh_2019', 'pct_pobreza_total']

# 1. Merge para SIEN (ATENCIÓN: MINSA lo llama 'UbigeoREN' pero usa códigos del INEI, cruzamos con 'inei')
for i, df_temp in enumerate(dfs_sien):
    df_temp['UbigeoREN'] = pd.to_numeric(df_temp['UbigeoREN'], errors='coerce')
    dfs_sien[i] = df_temp.merge(
        df_ubigeo[['inei'] + columnas_a_inyectar],
        left_on='UbigeoREN', right_on='inei', how='left'
    )

# 2. Merge para TAMIZAJE (Cruza usando id_ubigeo == inei)
df_tamizaje['id_ubigeo'] = pd.to_numeric(df_tamizaje['id_ubigeo'], errors='coerce')
df_tamizaje = df_tamizaje.merge(
    df_ubigeo[['inei'] + columnas_a_inyectar],
    left_on='id_ubigeo', right_on='inei', how='left'
)

# 3. Merge para ANEMIA_DA (Cruza por coincidencia de texto Dep/Prov/Dist)
df_da['DEPARTAMENTO'] = df_da['DEPARTAMENTO'].str.upper().str.strip()
df_da['PROVINCIA'] = df_da['PROVINCIA'].str.upper().str.strip()
df_da['DISTRITO'] = df_da['DISTRITO'].str.upper().str.strip()

df_da = df_da.merge(
    df_ubigeo[['departamento', 'provincia', 'distrito'] + columnas_a_inyectar],
    left_on=['DEPARTAMENTO', 'PROVINCIA', 'DISTRITO'],
    right_on=['departamento', 'provincia', 'distrito'],
    how='left'
)

print("Bases de datos enriquecidas con variables socioeconomicas y espaciales.")

## 7. Data Splitting: Construcción de Datasets Maestros
Para garantizar la calidad durante el entrenamiento del modelo, generamos dos ambientes de datos aplicando `.dropna()` de forma estricta sobre las características críticas, eliminando observaciones con ruido indescifrable.
1. **`df_regiones_enriquecido`:** Alta dimensionalidad de características (Solo Regiones).
2. **`df_global_limpio`:** Alta masividad de volumen de datos (Todas las fuentes).

In [ ]:
# ==========================================
# DATASET 1: df_regiones_enriquecido
# ==========================================
df_regiones_enriquecido = pd.concat(dfs_sien, ignore_index=True)
df_regiones_enriquecido = df_regiones_enriquecido.dropna(subset=['Target']) # Eliminar filas sin clasificación

# ==========================================
# DATASET 2: df_global_limpio
# ==========================================
columnas_comunes = ['EdadMeses', 'Sexo', 'Target'] + columnas_a_inyectar

# Extraemos estrictamente las columnas comunes de cada set antes de unirlos
df_sien_common = df_regiones_enriquecido[[c for c in columnas_comunes if c in df_regiones_enriquecido.columns]]
df_da_common = df_da[[c for c in columnas_comunes if c in df_da.columns]]
df_tam_common = df_tamizaje[[c for c in columnas_comunes if c in df_tamizaje.columns]]

df_global_limpio = pd.concat([df_sien_common, df_da_common, df_tam_common], ignore_index=True)
df_global_limpio = df_global_limpio.dropna(subset=columnas_comunes) # Limpieza estricta de Nulos para ML Base

# --- NUEVO: FEATURE SELECTION (PODA DE COLUMNAS) ---
# Eliminamos columnas crudas (IDs, nombres, fechas) que ya fueron explotadas
columnas_basura = [
    'Microred', 'EESSS', 'Dist_EESS', 'UbigeoPN', 'ProvinciaPN', 'DistritoPN', 
    'CentroPobladoPN', 'FechaHemoglobina', 'FechaAtencion', 'FechaNacimiento', 
    'DistritoREN', 'UbigeoREN', 'DISTRITO', 'PROVINCIA', 'DEPARTAMENTO', 'id_ubigeo'
]
df_regiones_enriquecido = df_regiones_enriquecido.drop(columns=columnas_basura, errors='ignore')

print(f"Dataset Regional Enriquecido creado. Shape: {df_regiones_enriquecido.shape}")
print(f"Dataset Global Limpio creado. Shape: {df_global_limpio.shape}")

## 8. Exportación Final
Los datos preprocesados son guardados para servir como entrada directa a la siguiente fase de Entrenamiento de Modelos.

In [ ]:
print("Iniciando exportación de datasets listos para Machine Learning...")

ruta_regiones = PROCESSED_DIR / 'df_regiones_enriquecido.csv'
ruta_global = PROCESSED_DIR / 'df_global_limpio.csv'

df_regiones_enriquecido.to_csv(ruta_regiones, index=False)
df_global_limpio.to_csv(ruta_global, index=False)

print(f"EXITO: {ruta_regiones.name} exportado correctamente.")
print(f"EXITO: {ruta_global.name} exportado correctamente.")